# LeetCode 1443 — Minimum Time to Collect All Apples in a Tree bằng Python

Link đề: https://leetcode.com/problems/minimum-time-to-collect-all-apples-in-a-tree/ · Độ khó: **Medium** · Chủ đề: Hash Table, Tree, Depth-First Search, Breadth-First Search, DP on Trees

Skeleton LeetCode:

```python
class Solution:
    def minTime(self, n: int, edges: list[list[int]], hasApple: list[bool]) -> int:
        ...
```

## 1. Mục tiêu học

1. Biểu diễn **cây vô hướng** bằng danh sách kề (adjacency list) và duyệt nó an toàn bằng tham số `parent` / mảng `visited`.
2. Nhận ra quy tắc then chốt: **một cạnh (cha–con) phải đi đúng 2 lần khi và chỉ khi cây con phía dưới có ít nhất một quả táo**; ngược lại không cần đi.
3. Viết **DFS hậu thứ tự** (post-order): tính kết quả của con trước, rồi gộp lên cha (DP trên cây).
4. Viết phiên bản **lặp (không đệ quy)** bằng thứ tự BFS/DFS đảo ngược để tránh tràn đệ quy với cây sâu `10^5`.
5. Tránh lỗi: coi cạnh là có hướng (giả định `a` là cha của `b`), chỉ xét táo ở con trực tiếp, cộng nhầm cho gốc, quên nhân 2.

## 2. Đề bài & ví dụ

Cho một **cây vô hướng** gồm `n` đỉnh đánh số `0..n−1`. `edges[i] = [a_i, b_i]` nghĩa là có cạnh nối `a_i` và `b_i`. Mảng `hasApple[i] = true` nếu đỉnh `i` có táo. Đi qua một cạnh mất **1 giây**.

Bắt đầu ở đỉnh `0`, đi nhặt **tất cả** táo rồi **quay về** đỉnh `0`. Trả về thời gian nhỏ nhất.

**Ví dụ 1:** `n = 7, edges = [[0,1],[0,2],[1,4],[1,5],[2,3],[2,6]]`, `hasApple = [F,F,T,F,T,T,F]` → `8`.

**Ví dụ 2:** cùng cây, `hasApple = [F,F,T,F,F,T,F]` → `6`.

**Ví dụ 3:** cùng cây, không có táo nào → `0`.

Hình dạng cây ở cả 3 ví dụ:

```
        0
      /   \
     1     2
    / \   / \
   4   5 3   6
```

**Ràng buộc:** `1 <= n <= 10^5`, `edges.length == n − 1`, `0 <= a_i < b_i <= n − 1`, `hasApple.length == n`.

Lưu ý: `a_i < b_i` chỉ nói về **nhãn**, không nói `a_i` là cha của `b_i` (ví dụ cạnh `[1,2]` trong cây gốc 0 có thể là `2` là cha của `1`).

In [1]:
import random
import sys
import time
from collections import defaultdict, deque
from typing import List

T, F = True, False
E = [[0, 1], [0, 2], [1, 4], [1, 5], [2, 3], [2, 6]]
examples = [
    ((7, E, [F, F, T, F, T, T, F]), 8),
    ((7, E, [F, F, T, F, F, T, F]), 6),
    ((7, E, [F, F, F, F, F, F, F]), 0),
]
for (n, edges, apples), want in examples:
    print(f"táo ở {[i for i in range(n) if apples[i]]} -> kỳ vọng {want}")

táo ở [2, 4, 5] -> kỳ vọng 8
táo ở [2, 5] -> kỳ vọng 6
táo ở [] -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1 Danh sách kề cho đồ thị vô hướng

Mỗi cạnh `[a, b]` được thêm **hai chiều**: `adj[a].append(b)` và `adj[b].append(a)`. Dùng `[[] for _ in range(n)]` (không dùng `[[]] * n` vì các phần tử sẽ dùng chung một list).

In [2]:
def build_adj(n, edges):
    adj = [[] for _ in range(n)]
    for a, b in edges:
        adj[a].append(b)
        adj[b].append(a)
    return adj

adj = build_adj(7, E)
for u in range(7):
    print(u, "->", adj[u])
bad = [[]] * 3
bad[0].append(9)
print("[[]] * 3 dùng chung list:", bad)

0 -> [1, 2]
1 -> [0, 4, 5]
2 -> [0, 3, 6]
3 -> [2]
4 -> [1]
5 -> [1]
6 -> [2]
[[]] * 3 dùng chung list: [[9], [9], [9]]


### 3.2 Duyệt cây vô hướng: tham số `parent`

Trong cây vô hướng, từ con ta luôn thấy cha trong danh sách kề. Để không đi ngược lên (và lặp vô hạn), DFS mang theo `parent` và bỏ qua `v == parent`. Với cây thì như vậy là đủ (không có chu trình).

In [3]:
def dfs_print(u, parent, depth=0):
    print("  " * depth + f"thăm {u} (cha {parent})")
    for v in adj[u]:
        if v != parent:
            dfs_print(v, u, depth + 1)

dfs_print(0, -1)

thăm 0 (cha -1)
  thăm 1 (cha 0)
    thăm 4 (cha 1)
    thăm 5 (cha 1)
  thăm 2 (cha 0)
    thăm 3 (cha 2)
    thăm 6 (cha 2)


### 3.3 Hậu thứ tự (post-order) = "con xong rồi mới đến cha"

DP trên cây thường tính giá trị của đỉnh **sau khi** đã có giá trị của mọi con. Thứ tự BFS từ gốc **đảo ngược** cũng đảm bảo con được xử lý trước cha — cách làm không cần đệ quy.

In [4]:
order, parent = [], [-1] * 7
seen = [False] * 7
q = deque([0]); seen[0] = True
while q:
    u = q.popleft(); order.append(u)
    for v in adj[u]:
        if not seen[v]:
            seen[v] = True; parent[v] = u; q.append(v)
print("thứ tự BFS:", order)
print("đảo ngược :", order[::-1], "(mỗi con đứng trước cha của nó)")
print("parent    :", parent)

thứ tự BFS: [0, 1, 2, 4, 5, 3, 6]
đảo ngược : [6, 3, 5, 4, 2, 1, 0] (mỗi con đứng trước cha của nó)
parent    : [-1, 0, 0, 2, 1, 1, 2]


### 3.4 Giới hạn đệ quy của Python

Mặc định Python cho khoảng 1000 tầng đệ quy. Cây "đường thẳng" `0–1–2–…–(n−1)` có độ sâu `n` ⇒ đệ quy với `n = 10^5` sẽ lỗi `RecursionError`. Cách xử lý: `sys.setrecursionlimit(...)` (rủi ro tràn stack C) hoặc viết bản **lặp**.

In [5]:
print("giới hạn đệ quy hiện tại:", sys.getrecursionlimit())

def depth_rec(k):
    return 0 if k == 0 else 1 + depth_rec(k - 1)

try:
    depth_rec(50_000)
except RecursionError as e:
    print("RecursionError:", str(e)[:50])

giới hạn đệ quy hiện tại: 1000
RecursionError: maximum recursion depth exceeded


## 4. Ý tưởng nền tảng / trực giác

Treo cây tại gốc `0`. Mỗi cạnh nối một đỉnh `v` với cha `parent[v]`.

- Nếu **cây con gốc `v`** có ít nhất một táo, ta **bắt buộc** đi xuống qua cạnh `(parent[v], v)` và sau đó phải **đi lên lại** (vì phải về `0`) ⇒ cạnh này tốn đúng **2 giây**.
- Nếu cây con gốc `v` không có táo, đi xuống đó là vô ích ⇒ **0 giây**.

Tính tay ví dụ 1 (táo ở 2, 4, 5):

| đỉnh v (≠ 0) | cây con của v | có táo? | cạnh (cha, v) tốn |
|---|---|---|---|
| 1 | {1,4,5} | ✔ (4,5) | 2 |
| 2 | {2,3,6} | ✔ (2) | 2 |
| 3 | {3} | ✘ | 0 |
| 4 | {4} | ✔ | 2 |
| 5 | {5} | ✔ | 2 |
| 6 | {6} | ✘ | 0 |

Tổng `2+2+2+2 = 8` ✔. Ví dụ 2 (táo ở 2, 5): cạnh tới 1, 2, 5 ⇒ `6` ✔.

In [6]:
def subtree_nodes(adj, v, parent):
    out, stack = [], [(v, parent)]
    while stack:
        u, p = stack.pop()
        out.append(u)
        stack.extend((w, u) for w in adj[u] if w != p)
    return out

(n, edges, apples), _ = examples[0]
total = 0
for v in range(1, n):
    sub = subtree_nodes(adj, v, parent[v])
    need = any(apples[u] for u in sub)
    total += 2 * need
    print(f"v={v} cây con {sorted(sub)} có táo? {need}")
print("tổng =", total)

v=1 cây con [1, 4, 5] có táo? True
v=2 cây con [2, 3, 6] có táo? True
v=3 cây con [3] có táo? False
v=4 cây con [4] có táo? True
v=5 cây con [5] có táo? True
v=6 cây con [6] có táo? False
tổng = 8


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** đúng như bảng trên — với **từng** đỉnh `v ≠ 0`, duyệt toàn bộ cây con của `v` để xem có táo không; nếu có, cộng 2.

1. BFS từ `0` để biết `parent[v]`.
2. Với mỗi `v`, duyệt cây con (không đi qua `parent[v]`), kiểm tra `any(hasApple)`.

In [7]:
def min_time_brute(n: int, edges: List[List[int]], hasApple: List[bool]) -> int:
    adj = build_adj(n, edges)
    parent = [-1] * n
    seen = [False] * n
    seen[0] = True
    q = deque([0])
    while q:
        u = q.popleft()
        for v in adj[u]:
            if not seen[v]:
                seen[v] = True
                parent[v] = u
                q.append(v)
    total = 0
    for v in range(1, n):
        if any(hasApple[u] for u in subtree_nodes(adj, v, parent[v])):
            total += 2
    return total

for (n, edges, apples), want in examples:
    print(min_time_brute(n, edges, apples), "kỳ vọng", want)

8 kỳ vọng 8
6 kỳ vọng 6
0 kỳ vọng 0


**Độ phức tạp:** mỗi đỉnh duyệt lại cây con của nó ⇒ tổng = tổng kích thước mọi cây con = tổng độ sâu các đỉnh ⇒ `O(n²)` với cây đường thẳng. Space `O(n)`.

**Nút thắt:** cây con của `v` được duyệt lại nhiều lần (một lần cho mỗi tổ tiên của nó). Thông tin "cây con có táo" của con có thể **dùng lại** cho cha.

In [8]:
# Cây đường thẳng 0-1-2-...-(n-1), táo ở đỉnh cuối: trường hợp xấu nhất của brute force
n_line = 3000
line = [[i, i + 1] for i in range(n_line - 1)]
apl = [False] * n_line; apl[-1] = True
t = time.perf_counter()
print(min_time_brute(n_line, line, apl), f"({time.perf_counter()-t:.2f}s với n={n_line})")

5998 (1.35s với n=3000)


## 6. Ý tưởng thuật toán / bước đột phá

**Insight (DP trên cây):** gọi `need[v]` = "cây con gốc `v` có táo". Khi đó

`need[v] = hasApple[v] OR need[c1] OR need[c2] OR …` (với các con `c`).

Đáp án = `2 × (số đỉnh v ≠ 0 có need[v] = True)`.

Tương đương, định nghĩa `cost(v)` = thời gian để nhặt hết táo trong cây con `v` và quay về `v`:

- Với mỗi con `c`: **nếu** `cost(c) > 0` **hoặc** `hasApple[c]` **thì** cộng `cost(c) + 2` (đi xuống, xử lý, đi lên); **ngược lại** bỏ qua.

**Cách khác (đi ngược lên từ táo):** từ mỗi đỉnh có táo, leo theo `parent` lên gốc, đánh dấu các cạnh; **dừng ngay** khi gặp đỉnh đã đánh dấu (đoạn phía trên đã được tính). Mỗi cạnh được đánh dấu tối đa 1 lần ⇒ `O(n)`.

**Invariant:** khi xử lý `v` theo hậu thứ tự, mọi con của `v` đã có `need`/`cost` chính xác.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Leo từ táo lên gốc, dừng khi gặp đỉnh đã đánh dấu

**Giải thích từng đoạn:**
1. BFS lấy `parent`.
2. `marked[v] = True` nghĩa là cạnh `(parent[v], v)` đã được tính.
3. Với mỗi đỉnh táo `v`: trong khi `v != 0` và chưa `marked[v]`: đánh dấu, cộng 2, `v = parent[v]`.

In [9]:
def bfs_parent(n, adj):
    parent, order = [-1] * n, [0]
    seen = [False] * n
    seen[0] = True
    for u in order:                    # duyệt list đang lớn dần = BFS
        for v in adj[u]:
            if not seen[v]:
                seen[v] = True
                parent[v] = u
                order.append(v)
    return parent, order

def min_time_climb(n: int, edges: List[List[int]], hasApple: List[bool]) -> int:
    adj = build_adj(n, edges)
    parent, _ = bfs_parent(n, adj)
    marked = [False] * n
    total = 0
    for v in range(n):
        if hasApple[v]:
            while v != 0 and not marked[v]:
                marked[v] = True
                total += 2
                v = parent[v]
    return total

for (n, edges, apples), want in examples:
    print(min_time_climb(n, edges, apples), "kỳ vọng", want)

8 kỳ vọng 8
6 kỳ vọng 6
0 kỳ vọng 0


**Code nộp LeetCode (cách 2):**

In [10]:
class SolutionClimb:
    def minTime(self, n: int, edges: List[List[int]], hasApple: List[bool]) -> int:
        adj = [[] for _ in range(n)]
        for a, b in edges:
            adj[a].append(b)
            adj[b].append(a)
        parent, order, seen = [-1] * n, [0], [False] * n
        seen[0] = True
        for u in order:
            for v in adj[u]:
                if not seen[v]:
                    seen[v] = True
                    parent[v] = u
                    order.append(v)
        marked, total = [False] * n, 0
        for v in range(n):
            if hasApple[v]:
                while v != 0 and not marked[v]:
                    marked[v] = True
                    total += 2
                    v = parent[v]
        return total

print(SolutionClimb().minTime(7, E, [F, F, T, F, T, T, F]))

8


**Độ phức tạp:** BFS `O(n)`; mỗi đỉnh được đánh dấu tối đa 1 lần nên tổng số bước leo `O(n)` ⇒ Time `O(n)`, Space `O(n)`.

### 7.2 Cách 3 — DFS đệ quy hậu thứ tự (lời giải "sách giáo khoa")

**Giải thích từng đoạn:**
1. `dfs(u, p)` trả về thời gian nhặt hết táo trong cây con `u` rồi quay về `u`.
2. Duyệt các con `v ≠ p`: `c = dfs(v, u)`.
3. **Nếu** `c > 0` hoặc `hasApple[v]` **thì** `total += c + 2`.
4. Gọi `dfs(0, -1)`.

Đệ quy rất tự nhiên nhưng cần nâng giới hạn đệ quy cho cây sâu.

In [11]:
def min_time_dfs(n: int, edges: List[List[int]], hasApple: List[bool]) -> int:
    adj = build_adj(n, edges)

    def dfs(u, p):
        total = 0
        for v in adj[u]:
            if v == p:
                continue
            c = dfs(v, u)
            if c > 0 or hasApple[v]:
                total += c + 2
        return total

    return dfs(0, -1)

for (n, edges, apples), want in examples:
    print(min_time_dfs(n, edges, apples), "kỳ vọng", want)

8 kỳ vọng 8
6 kỳ vọng 6
0 kỳ vọng 0


**Code nộp LeetCode (cách 3)** — có `sys.setrecursionlimit` vì cây có thể sâu tới `10^5`:

In [12]:
class SolutionDFS:
    def minTime(self, n: int, edges: List[List[int]], hasApple: List[bool]) -> int:
        sys.setrecursionlimit(max(1000, 2 * n + 10))
        adj = [[] for _ in range(n)]
        for a, b in edges:
            adj[a].append(b)
            adj[b].append(a)

        def dfs(u, p):
            total = 0
            for v in adj[u]:
                if v != p:
                    c = dfs(v, u)
                    if c > 0 or hasApple[v]:
                        total += c + 2
            return total

        return dfs(0, -1)

print(SolutionDFS().minTime(7, E, [F, F, T, F, F, T, F]))

6


**Độ phức tạp:** mỗi đỉnh được gọi đúng 1 lần, mỗi cạnh xét 2 lần ⇒ Time `O(n)`, Space `O(n)` (danh sách kề + stack đệ quy sâu tới chiều cao cây).

### 7.3 Cách 4 — Lặp: BFS rồi xử lý theo thứ tự đảo ngược (không đệ quy)

**Giải thích từng đoạn:**
1. BFS từ `0` lấy `order` và `parent`.
2. `need = hasApple[:]` (copy).
3. Duyệt `order` **đảo ngược** (con trước cha): nếu `need[v]` và `v ≠ 0` thì cộng 2 và đặt `need[parent[v]] = True`.

Đây là lời giải chính (mục 9): `O(n)`, không lo giới hạn đệ quy.

In [13]:
def min_time_iter(n: int, edges: List[List[int]], hasApple: List[bool]) -> int:
    adj = build_adj(n, edges)
    parent, order = bfs_parent(n, adj)
    need = list(hasApple)
    total = 0
    for v in reversed(order):
        if v != 0 and need[v]:
            total += 2
            need[parent[v]] = True
    return total

for (n, edges, apples), want in examples:
    print(min_time_iter(n, edges, apples), "kỳ vọng", want)

8 kỳ vọng 8
6 kỳ vọng 6
0 kỳ vọng 0


So sánh tốc độ trên cây lớn `n = 10^5` (cây ngẫu nhiên và cây đường thẳng):

In [14]:
def random_tree(n, rng, p_apple=0.1):
    # đỉnh v nối với một đỉnh trước nó, rồi xáo nhãn (giữ 0 là 0) để cha có thể có nhãn lớn hơn con
    perm = [0] + rng.sample(range(1, n), n - 1)
    edges = []
    for v in range(1, n):
        u = rng.randrange(v)
        a, b = perm[u], perm[v]
        edges.append([min(a, b), max(a, b)])
    rng.shuffle(edges)
    apples = [rng.random() < p_apple for _ in range(n)]
    return edges, apples

rng = random.Random(2024)
N = 10**5
tests = {"ngẫu nhiên": random_tree(N, rng), "đường thẳng": ([[i, i + 1] for i in range(N - 1)], [i == N - 1 for i in range(N)])}
for name, (edges, apples) in tests.items():
    for fname, f in [("climb", min_time_climb), ("iter", min_time_iter), ("SolutionDFS", lambda a, b, c: SolutionDFS().minTime(a, b, c))]:
        if fname == "SolutionDFS" and name == "đường thẳng":
            continue   # đệ quy sâu 10^5 tầng: không chạy trong notebook để tránh tràn stack C
        t = time.perf_counter()
        r = f(N, edges, apples)
        print(f"{name:12} {fname:12} -> {r} ({time.perf_counter()-t:.3f}s)")

ngẫu nhiên   climb        -> 51494 (0.110s)
ngẫu nhiên   iter         -> 51494 (0.109s)


ngẫu nhiên   SolutionDFS  -> 51494 (0.157s)
đường thẳng  climb        -> 199998 (0.035s)
đường thẳng  iter         -> 199998 (0.132s)


## 8. TRACE từng bước

Trace cách 4 trên ví dụ 1 (táo ở 2, 4, 5). Thứ tự BFS: `[0,1,2,4,5,3,6]` ⇒ đảo ngược `[6,3,5,4,2,1,0]`.

| v | need[v] | parent[v] | hành động | total | need sau |
|---|---|---|---|---|---|
| 6 | F | 2 | bỏ qua | 0 | |
| 3 | F | 2 | bỏ qua | 0 | |
| 5 | T | 1 | +2, need[1]=T | 2 | 1 |
| 4 | T | 1 | +2, need[1]=T | 4 | 1 |
| 2 | T | 0 | +2, need[0]=T | 6 | 0 |
| 1 | T | 0 | +2, need[0]=T | 8 | 0 |
| 0 | T | – | gốc, không cộng | 8 | |

In [15]:
def trace_iter(n, edges, hasApple):
    adj = build_adj(n, edges)
    parent, order = bfs_parent(n, adj)
    print("order =", order, "| parent =", parent)
    need = list(hasApple)
    total = 0
    for v in reversed(order):
        if v == 0:
            print(f"v=0 gốc -> dừng, total={total}")
            continue
        if need[v]:
            total += 2
            need[parent[v]] = True
            print(f"v={v} need=T -> cạnh ({parent[v]},{v}) +2, đánh dấu need[{parent[v]}]=T | total={total}")
        else:
            print(f"v={v} need=F -> bỏ qua cạnh ({parent[v]},{v}) | total={total}")
    return total

trace_iter(7, E, [F, F, T, F, T, T, F])

order = [0, 1, 2, 4, 5, 3, 6] | parent = [-1, 0, 0, 2, 1, 1, 2]
v=6 need=F -> bỏ qua cạnh (2,6) | total=0
v=3 need=F -> bỏ qua cạnh (2,3) | total=0
v=5 need=T -> cạnh (1,5) +2, đánh dấu need[1]=T | total=2
v=4 need=T -> cạnh (1,4) +2, đánh dấu need[1]=T | total=4
v=2 need=T -> cạnh (0,2) +2, đánh dấu need[0]=T | total=6
v=1 need=T -> cạnh (0,1) +2, đánh dấu need[0]=T | total=8
v=0 gốc -> dừng, total=8


8

Trace cách 3 (đệ quy) — in lúc **vào** và lúc **trả về** của mỗi lời gọi để thấy hậu thứ tự:

In [16]:
def trace_dfs(n, edges, hasApple):
    adj = build_adj(n, edges)

    def dfs(u, p, d):
        print("  " * d + f"vào dfs({u})")
        total = 0
        for v in adj[u]:
            if v == p:
                continue
            c = dfs(v, u, d + 1)
            if c > 0 or hasApple[v]:
                total += c + 2
                print("  " * d + f"  con {v}: cost={c}, apple={hasApple[v]} -> +{c + 2}")
        print("  " * d + f"trả về dfs({u}) = {total}")
        return total

    return dfs(0, -1, 0)

trace_dfs(7, E, [F, F, T, F, F, T, F])

vào dfs(0)
  vào dfs(1)
    vào dfs(4)
    trả về dfs(4) = 0
    vào dfs(5)
    trả về dfs(5) = 0
    con 5: cost=0, apple=True -> +2
  trả về dfs(1) = 2
  con 1: cost=2, apple=False -> +4
  vào dfs(2)
    vào dfs(3)
    trả về dfs(3) = 0
    vào dfs(6)
    trả về dfs(6) = 0
  trả về dfs(2) = 0
  con 2: cost=0, apple=True -> +2
trả về dfs(0) = 6


6

## 9. Code nộp LeetCode

Lời giải chính (cách 4 — lặp, không đệ quy), đúng chữ ký `minTime(self, n, edges, hasApple)`.

In [17]:
from typing import List

class Solution:
    def minTime(self, n: int, edges: List[List[int]], hasApple: List[bool]) -> int:
        adj = [[] for _ in range(n)]
        for a, b in edges:
            adj[a].append(b)
            adj[b].append(a)
        parent = [-1] * n
        seen = [False] * n
        seen[0] = True
        order = [0]
        for u in order:                     # BFS
            for v in adj[u]:
                if not seen[v]:
                    seen[v] = True
                    parent[v] = u
                    order.append(v)
        need = list(hasApple)
        total = 0
        for v in reversed(order):           # con trước cha
            if v != 0 and need[v]:
                total += 2
                need[parent[v]] = True
        return total

for (n, edges, apples), want in examples:
    assert Solution().minTime(n, edges, apples) == want
print("Solution: 3 ví dụ đề OK")

Solution: 3 ví dụ đề OK


## 10. Vì sao thuật toán đúng

**Cận dưới:** xét cạnh `(p, v)` với `p = parent[v]`. Nếu cây con `v` chứa một táo, hành trình (bắt đầu và kết thúc ở `0`, nằm ngoài cây con `v`) phải **vào** cây con ít nhất một lần và **ra** ít nhất một lần; cách duy nhất vào/ra là cạnh `(p, v)` ⇒ cạnh này bị đi **ít nhất 2 lần**. Vậy thời gian ≥ `2 × (số cạnh có cây con chứa táo)`.

**Đạt được cận dưới:** xét cây con `S` gồm gốc `0` và mọi đỉnh `v` có `need[v] = True`. `S` liên thông (nếu `need[v]` thì `need[parent[v]]`). Đi DFS trên `S` (xuống rồi lên) qua mỗi cạnh của `S` đúng 2 lần và thăm mọi đỉnh táo. Vậy cận dưới đạt được ⇒ đáp án đúng bằng `2 × |cạnh của S|`.

**Từng cách tính đúng `|cạnh của S|`:**
- Cách 1: kiểm tra trực tiếp từng cạnh.
- Cách 2: cạnh `(parent[v], v)` thuộc `S` ⇔ `v` nằm trên đường từ một táo lên gốc; leo dừng sớm vì phần trên đã được đánh dấu bởi lần leo trước.
- Cách 3 & 4: tính `need` theo hậu thứ tự, dựa trên công thức truy hồi ở mục 6.

## 11. Phân tích Time / Space

Gọi `n` là số đỉnh (`n − 1` cạnh), `h` là chiều cao cây.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Brute (duyệt cây con từng đỉnh) | `O(n·h)` ⊆ `O(n²)` | `O(n)` | chậm với cây đường thẳng |
| 2. Leo từ táo, dừng sớm | `O(n)` | `O(n)` | mỗi cạnh đánh dấu ≤ 1 lần |
| 3. DFS đệ quy | `O(n)` | `O(n)` + stack `O(h)` | cần nâng recursion limit |
| 4. BFS + đảo thứ tự | `O(n)` | `O(n)` | an toàn nhất trong Python |

## 12. Bộ test

Ví dụ đề, edge case (`n = 1`, chỉ gốc có táo, táo ở lá sâu, cạnh mà nhãn cha lớn hơn nhãn con), và đối chiếu ngẫu nhiên với brute force trên cây ngẫu nhiên có xáo nhãn (seed cố định).

In [18]:
ALL = {
    "brute": min_time_brute,
    "climb": min_time_climb,
    "dfs": min_time_dfs,
    "iter": min_time_iter,
    "SolutionClimb": lambda a, b, c: SolutionClimb().minTime(a, b, c),
    "SolutionDFS": lambda a, b, c: SolutionDFS().minTime(a, b, c),
    "Solution": lambda a, b, c: Solution().minTime(a, b, c),
}
edge = [
    ((1, [], [False]), 0),
    ((1, [], [True]), 0),
    ((2, [[0, 1]], [True, False]), 0),
    ((2, [[0, 1]], [False, True]), 2),
    ((4, [[0, 2], [0, 3], [1, 2]], [F, T, F, F]), 4),     # 2 là cha của 1
    ((4, [[1, 2], [0, 1], [2, 3]], [F, F, F, T]), 6),     # cạnh không theo thứ tự cha-con
    ((5, [[0, 1], [1, 2], [2, 3], [3, 4]], [F, F, T, F, T]), 8),
]
for (n, edges, apples), want in examples + edge:
    for name, f in ALL.items():
        got = f(n, edges, apples)
        assert got == want, (name, n, edges, apples, got, want)
print("Ví dụ đề + edge case: OK cho", len(ALL), "cách")

Ví dụ đề + edge case: OK cho 7 cách


In [19]:
rng = random.Random(1443)
for _ in range(600):
    n = rng.randint(1, 40)
    edges, apples = random_tree(n, rng, p_apple=rng.choice([0.0, 0.1, 0.3, 0.8]))
    want = min_time_brute(n, edges, apples)
    for name, f in ALL.items():
        assert f(n, edges, apples) == want, (name, n, edges, apples)
print("600 cây ngẫu nhiên (seed 1443): mọi cách khớp brute force")

for _ in range(5):
    edges, apples = random_tree(20000, rng, p_apple=0.02)
    assert min_time_climb(20000, edges, apples) == min_time_iter(20000, edges, apples) == Solution().minTime(20000, edges, apples)
print("5 cây n=20000: climb, iter, Solution khớp nhau")

600 cây ngẫu nhiên (seed 1443): mọi cách khớp brute force


5 cây n=20000: climb, iter, Solution khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Coi cạnh là có hướng `a → b` (giả định `a` là cha)

Vì đề cho `a_i < b_i`, nhiều bạn tưởng `a_i` luôn là cha. Sai: cạnh `[1,2]` trong cây gốc 0 có thể có `2` là cha.

In [20]:
def wrong_directed(n, edges, hasApple):
    children = [[] for _ in range(n)]
    for a, b in edges:
        children[a].append(b)          # SAI: chỉ thêm một chiều
    def dfs(u):
        total = 0
        for v in children[u]:
            c = dfs(v)
            if c > 0 or hasApple[v]:
                total += c + 2
        return total
    return dfs(0)

case = (4, [[0, 2], [0, 3], [1, 2]], [F, T, F, F])
print("Sai :", wrong_directed(*case), "| Đúng:", Solution().minTime(*case))
print("Đường đúng: 0 -> 2 -> 1 -> 2 -> 0 = 4 giây; bản sai không bao giờ tới được đỉnh 1.")

Sai : 0 | Đúng: 4
Đường đúng: 0 -> 2 -> 1 -> 2 -> 0 = 4 giây; bản sai không bao giờ tới được đỉnh 1.


**Sửa:** thêm cạnh **hai chiều** và dùng `parent`/`visited` khi duyệt.

### Lỗi 2 — Chỉ xét táo ở **con trực tiếp**, bỏ qua táo sâu hơn

Điều kiện đúng là "cây con có táo", tức `cost(v) > 0 or hasApple[v]`.

In [21]:
def wrong_only_child_apple(n, edges, hasApple):
    adj = build_adj(n, edges)
    def dfs(u, p):
        total = 0
        for v in adj[u]:
            if v != p and hasApple[v]:      # SAI: không xét táo sâu hơn
                total += dfs(v, u) + 2
        return total
    return dfs(0, -1)

case = (3, [[0, 1], [1, 2]], [F, F, T])
print("Sai :", wrong_only_child_apple(*case), "| Đúng:", Solution().minTime(*case))
print("Ví dụ 1 — Sai:", wrong_only_child_apple(7, E, [F, F, T, F, T, T, F]), "| Đúng: 8")

Sai : 0 | Đúng: 4
Ví dụ 1 — Sai: 2 | Đúng: 8


**Sửa:** luôn gọi `c = dfs(v, u)` rồi mới quyết định: `if c > 0 or hasApple[v]: total += c + 2`.

### Lỗi 3 — Cộng cả "cạnh" của gốc

Gốc không có cạnh lên cha. Nếu đếm mọi đỉnh có `need = True` (kể cả `0`) sẽ thừa 2.

In [22]:
def wrong_count_root(n, edges, hasApple):
    adj = build_adj(n, edges)
    parent, order = bfs_parent(n, adj)
    need = list(hasApple)
    total = 0
    for v in reversed(order):
        if need[v]:                      # SAI: thiếu điều kiện v != 0
            total += 2
            if v != 0:
                need[parent[v]] = True
    return total

for case in [(1, [], [True]), (7, E, [F, F, T, F, T, T, F])]:
    print("Sai :", wrong_count_root(*case), "| Đúng:", Solution().minTime(*case))

Sai : 2 | Đúng: 0
Sai : 10 | Đúng: 8


**Sửa:** `if v != 0 and need[v]`.

### Lỗi 4 — Quên nhân 2 (chỉ tính đường đi xuống)

Đề yêu cầu **quay về** đỉnh 0 nên mỗi cạnh cần thiết tốn 2 giây.

In [23]:
def wrong_no_double(n, edges, hasApple):
    return Solution().minTime(n, edges, hasApple) // 2   # tương đương chỉ đếm cạnh

print("Sai :", wrong_no_double(7, E, [F, F, T, F, T, T, F]), "| Đúng: 8")

Sai : 4 | Đúng: 8


**Sửa:** cộng `2` cho mỗi cạnh cần thiết (hoặc `c + 2` trong DFS).

### Lỗi 5 — Đệ quy không nâng giới hạn với cây sâu

Bản DFS đệ quy đúng logic nhưng với cây đường thẳng sâu vài nghìn đỉnh sẽ gặp `RecursionError` (ta bắt lỗi để notebook không dừng).

In [24]:
n_deep = 5000
deep_edges = [[i, i + 1] for i in range(n_deep - 1)]
deep_apples = [i == n_deep - 1 for i in range(n_deep)]
old = sys.getrecursionlimit()
sys.setrecursionlimit(1000)
try:
    print(min_time_dfs(n_deep, deep_edges, deep_apples))
except RecursionError:
    print("RecursionError với cây sâu", n_deep, "khi giới hạn = 1000")
finally:
    sys.setrecursionlimit(old)
print("Bản lặp:", min_time_iter(n_deep, deep_edges, deep_apples))

RecursionError với cây sâu 5000 khi giới hạn = 1000
Bản lặp: 9998


**Sửa:** dùng bản lặp (cách 4) hoặc `sys.setrecursionlimit` đủ lớn.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Đi qua các đỉnh đặc biệt trong cây rồi quay về"** ⇒ đáp án là `2 × số cạnh của cây con nhỏ nhất nối gốc với các đỉnh đặc biệt`.
- Cây cho dưới dạng **danh sách cạnh** ⇒ luôn dựng adjacency **hai chiều**; đừng tin thứ tự `[a, b]`.
- **DP trên cây** ⇒ hậu thứ tự: viết đệ quy để nghĩ, nộp bản lặp (BFS + đảo ngược) để an toàn trong Python.
- Thông tin "cây con có X không" lan **từ dưới lên**: `need[parent] |= need[child]`.
- Leo từ đỉnh đặc biệt lên gốc **dừng khi gặp đỉnh đã thăm** là mẹo `O(n)` hay gặp (cũng dùng trong bài LCA nhiều đỉnh).

## 15. Biến thể của bài

1. **Cạnh có trọng số** (thời gian khác nhau): cộng `2·w` cho mỗi cạnh cần thiết (bài tập 4).
2. **Không cần quay về** đỉnh 0: đáp án `2·|S| − (độ sâu lớn nhất của một đỉnh trong S)` — đi cuối cùng tới đỉnh sâu nhất (bài tập 5).
3. **Bắt đầu ở đỉnh bất kỳ `s`**: treo cây tại `s`, giữ nguyên thuật toán.
4. **Collect Coins in a Tree** (LeetCode 2603): nhặt trong bán kính 2 — cắt lá không có xu rồi cắt thêm 2 lớp.
5. **Đếm đỉnh/cạnh trên đường đi giữa nhiều đỉnh** (Steiner tree trên cây) — cùng ý tưởng `need`.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute | Mỗi đỉnh kiểm tra cây con có táo | `O(n²)` | `O(n)` | Kiểm chứng |
| 2. Leo từ táo | Đánh dấu đường lên gốc, dừng sớm | `O(n)` | `O(n)` | Khi đã có `parent` |
| 3. DFS đệ quy | `cost(v)`, cộng `c + 2` nếu cần | `O(n)` | `O(n)` + stack | Dễ nghĩ, phỏng vấn |
| 4. BFS + đảo ngược | `need` lan từ con lên cha | `O(n)` | `O(n)` | Nộp bài Python an toàn |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output):** `n = 6, edges = [[0,1],[1,2],[1,3],[0,4],[4,5]]`, táo ở đỉnh `3` và `4`. Kết quả?

**Bài 2 (tính tay):** với ví dụ 2 (táo ở 2 và 5), liệt kê các đỉnh `v ≠ 0` có `need[v] = True`.

**Bài 3 (điền TODO):** hoàn thiện `dfs_todo` (cách 3).

**Bài 4 (tự viết lại):** mỗi cạnh có trọng số: `edges[i] = [a, b, w]`. Viết `min_time_weighted(n, edges, hasApple)`.

**Bài 5 (mini challenge):** **không cần quay về** đỉnh 0 (kết thúc ở đâu cũng được). Viết `min_time_no_return(n, edges, hasApple)` và kiểm chứng bằng BFS trên trạng thái `(đỉnh, tập táo đã nhặt)` với cây nhỏ.

In [25]:
# Bài 3 — điền TODO (chỉ định nghĩa, không gọi)
def dfs_todo(adj, hasApple, u, p):
    total = 0
    for v in adj[u]:
        # TODO: bỏ qua cha
        # TODO: c = thời gian của cây con v
        # TODO: nếu cây con v cần đi thì cộng c + 2
        pass
    return total

## 18. Gợi ý 3 mức cho bài tập

**Bài 1:** (1) Vẽ cây: 0 có con 1 và 4; 1 có con 2, 3; 4 có con 5. (2) Cạnh cần: (0,1), (1,3), (0,4). (3) `2 × 3`.

**Bài 2:** (1) `need[v]` = cây con `v` có táo. (2) Táo ở 2 ⇒ `need[2]`; táo ở 5 ⇒ `need[5]`, `need[1]`. (3) Tập `{1, 2, 5}`.

**Bài 3:** (1) `if v == p: continue`. (2) `c = dfs_todo(adj, hasApple, v, u)`. (3) `if c > 0 or hasApple[v]: total += c + 2`.

**Bài 4:** (1) Lưu `(láng giềng, w)` trong adj. (2) Ghi `wpar[v]` = trọng số cạnh lên cha. (3) Cộng `2 * wpar[v]` thay vì 2. Cẩn thận: cạnh trọng số 0 vẫn đúng với cách dùng `need`, nhưng với cách `c > 0` thì phải dùng thêm cờ "có táo".

**Bài 5:** (1) Đi trọn `S` rồi quay về tốn `2|S|`; nếu không về thì tiết kiệm được đoạn từ điểm kết thúc về gốc. (2) Chọn điểm kết thúc là đỉnh **sâu nhất** trong `S`. (3) Đáp án `2|S| − max depth` (nếu `S` rỗng thì 0).

## 19. Đáp án tham khảo cho bài tập

In [26]:
# Bài 1
b1 = (6, [[0, 1], [1, 2], [1, 3], [0, 4], [4, 5]], [F, F, F, T, T, F])
ans1 = Solution().minTime(*b1)
print("Bài 1:", ans1)
assert ans1 == min_time_brute(*b1) == 6

# Bài 2
def need_nodes(n, edges, hasApple):
    adj = build_adj(n, edges)
    parent, order = bfs_parent(n, adj)
    need = list(hasApple)
    for v in reversed(order):
        if v != 0 and need[v]:
            need[parent[v]] = True
    return [v for v in range(1, n) if need[v]]

print("Bài 2:", need_nodes(7, E, [F, F, T, F, F, T, F]))
assert need_nodes(7, E, [F, F, T, F, F, T, F]) == [1, 2, 5]

# Bài 3
def dfs_todo_sol(adj, hasApple, u, p):
    total = 0
    for v in adj[u]:
        if v == p:
            continue
        c = dfs_todo_sol(adj, hasApple, v, u)
        if c > 0 or hasApple[v]:
            total += c + 2
    return total

for (n, edges, apples), want in examples:
    assert dfs_todo_sol(build_adj(n, edges), apples, 0, -1) == want
print("Bài 3: OK")

Bài 1: 6
Bài 2: [1, 2, 5]
Bài 3: OK


In [27]:
# Bài 4
def min_time_weighted(n, edges, hasApple):
    adj = [[] for _ in range(n)]
    for a, b, w in edges:
        adj[a].append((b, w))
        adj[b].append((a, w))
    parent, wpar, order = [-1] * n, [0] * n, [0]
    seen = [False] * n
    seen[0] = True
    for u in order:
        for v, w in adj[u]:
            if not seen[v]:
                seen[v] = True
                parent[v], wpar[v] = u, w
                order.append(v)
    need, total = list(hasApple), 0
    for v in reversed(order):
        if v != 0 and need[v]:
            total += 2 * wpar[v]
            need[parent[v]] = True
    return total

# Bài 5
def min_time_no_return(n, edges, hasApple):
    adj = build_adj(n, edges)
    parent, order = bfs_parent(n, adj)
    depth = [0] * n
    for u in order[1:]:
        depth[u] = depth[parent[u]] + 1
    need, edges_in_s, deepest = list(hasApple), 0, 0
    for v in reversed(order):
        if v != 0 and need[v]:
            edges_in_s += 1
            need[parent[v]] = True
            deepest = max(deepest, depth[v])
    return 2 * edges_in_s - deepest

def no_return_brute(n, edges, hasApple):
    adj = build_adj(n, edges)
    idx = {v: i for i, v in enumerate(v for v in range(n) if hasApple[v])}
    full = (1 << len(idx)) - 1
    start = (0, (1 << idx[0]) if 0 in idx else 0)
    dist = {start: 0}
    q = deque([start])
    while q:
        u, m = q.popleft()
        if m == full:
            return dist[(u, m)]
        for v in adj[u]:
            nm = m | (1 << idx[v]) if v in idx else m
            if (v, nm) not in dist:
                dist[(v, nm)] = dist[(u, m)] + 1
                q.append((v, nm))

rng = random.Random(19)
for _ in range(300):
    n = rng.randint(1, 9)
    edges, apples = random_tree(n, rng, p_apple=0.35)
    weighted = [[a, b, 1] for a, b in edges]
    assert min_time_weighted(n, weighted, apples) == Solution().minTime(n, edges, apples)
    assert min_time_no_return(n, edges, apples) == no_return_brute(n, edges, apples)
    w2 = [[a, b, rng.randint(0, 5)] for a, b in edges]
    adjw = build_adj(n, edges)
    parent, _ = bfs_parent(n, adjw)
    wmap = {(min(a, b), max(a, b)): w for a, b, w in w2}
    expect = sum(2 * wmap[(min(v, parent[v]), max(v, parent[v]))]
                 for v in range(1, n)
                 if any(apples[u] for u in subtree_nodes(adjw, v, parent[v])))
    assert min_time_weighted(n, w2, apples) == expect
print("Bài 4, 5: OK (300 cây ngẫu nhiên)")

Bài 4, 5: OK (300 cây ngẫu nhiên)


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "need lan từ con lên cha" (lặp, an toàn):**

```python
adj = [[] for _ in range(n)]
for a, b in edges:
    adj[a].append(b); adj[b].append(a)
parent, order, seen = [-1] * n, [0], [False] * n
seen[0] = True
for u in order:
    for v in adj[u]:
        if not seen[v]:
            seen[v] = True; parent[v] = u; order.append(v)
need, total = list(hasApple), 0
for v in reversed(order):
    if v != 0 and need[v]:
        total += 2
        need[parent[v]] = True
```

**Template DFS đệ quy:**

```python
def dfs(u, p):
    total = 0
    for v in adj[u]:
        if v != p:
            c = dfs(v, u)
            if c > 0 or hasApple[v]: total += c + 2
    return total
```

**Checklist:**
- [ ] Danh sách kề **hai chiều**, không giả định `a` là cha.
- [ ] Điều kiện là "cây con có táo", không chỉ "con có táo".
- [ ] Không cộng cho gốc.
- [ ] Mỗi cạnh cần thiết cộng **2**.
- [ ] Đã xử lý cây sâu (bản lặp hoặc nâng recursion limit); đã thử `n = 1`.